# Nautiq — Gold de escalas portuarias para JIT y ML

**Granularidad:** 1 fila = 1 `port_call`.

La lógica de detección se mantiene igual que en la versión validada:

- `SERVICE`: punto AIS real del puerto, radio máximo **0.5 NM**, `SOG <= 0.1 kn` y parada persistente.
- `ANCHOR`: únicamente antes del primer servicio, entre **0.5 y 5 NM**, con `status=1` o parada persistente `SOG <= 0.5 kn`.
- `DIRECT_TO_SERVICE`: seguimiento continuo hasta servicio sin fondeo previo, `actual_wait_hours = 0`.
- Casos incompletos se conservan en la Gold, pero `actual_wait_hours` queda `NULL` y no entran en ML/baselines.




In [0]:
%run ../setup_nautiq_dev

# Nautiq - setup del entorno

Configuración centralizada para el flujo activo del TFM:

- Acceso seguro a Bronze mediante SAS Token.
- Rutas de los topics AIS.
- Estado técnico de Auto Loader.
- Tablas Silver DEV.
- Gold histórico de port calls.
- Gold de baseline histórico de espera.
- Gold de predicciones JIT actuales.
- Modelo ML registrado en Unity Catalog con alias `Champion`.
- Cambio futuro entre tablas administradas y ADLS externo.

### Notebooks activos

- `silver_ais_positions_dev`
- `silver_ais_static_dev`
- `gold_vessel_port_calls_jit`
- `gold_waiting_avg_per_length`
- `ml_vessel_jit_classification`
- `gold_vessel_jit_current_predictions`


DataFrame[]

NAUTIQ - CONFIGURACION DEL ENTORNO
Environment: dev
Catalog: masterxyz002dbr
Target storage mode: managed
Ops volume: /Volumes/masterxyz002dbr/ops/nautiq_dev

[OK] ais_positions: 16 elementos encontrados
[OK] ais_static: 24 elementos encontrados

Silver DEV tables:
  - masterxyz002dbr.silver.ais_positions_dev
  - masterxyz002dbr.silver.ais_static_dev

Gold tables:
  - masterxyz002dbr.gold.vessel_port_calls_analytics
  - masterxyz002dbr.gold.waiting_avg_per_length
  - masterxyz002dbr.gold.vessel_jit_current_predictions

Registered ML model:
  - masterxyz002dbr.gold.vessel_jit_classifier@Champion

Active notebooks:
  - silver_ais_positions_dev
  - silver_ais_static_dev
  - gold_vessel_port_calls_jit
  - gold_waiting_avg_per_length
  - ml_vessel_jit_classification
  - gold_vessel_jit_current_predictions

[OK] Setup completado correctamente.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

spark.conf.set("spark.sql.session.timeZone", "UTC")

positions_table = positions_target_table
static_table = static_target_table
target_table = vessel_port_calls_target_table
target_path = vessel_port_calls_target_path

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{gold_schema}")

print("Positions:", positions_table)
print("Static:", static_table)
print("Gold:", target_table)


Positions: masterxyz002dbr.silver.ais_positions_dev
Static: masterxyz002dbr.silver.ais_static_dev
Gold: masterxyz002dbr.gold.vessel_port_calls_analytics


In [0]:
# ============================================================
# 1. PARAMETROS
# ============================================================

EARTH_RADIUS_NM = 3440.065
APPROACH_RADIUS_NM = 25.0
SERVICE_RADIUS_NM = 0.5
ANCHOR_RADIUS_NM = 5.0

SERVICE_MAX_SPEED_KNOTS = 0.1
ANCHOR_MAX_SPEED_KNOTS = 0.5
APPROACH_MIN_SPEED_KNOTS = 1.0

MIN_SERVICE_MINUTES = 10.0
MIN_ANCHOR_MINUTES = 10.0
MIN_SEGMENT_POSITIONS = 2
MAX_CONTINUOUS_GAP_MINUTES = 30.0
DIRECT_SERVICE_MAX_GAP_MINUTES = 30.0
PORT_CALL_GAP_HOURS = 12.0

MIN_DISTANCE_PROGRESS_NM = 0.02
MAX_COG_SERVICE_DIFF_DEG = 60.0

# Puntos de parada/servicio recurrentes encontrados en AIS.
# La configuración de los puertos se carga desde setup_nautiq_dev.

ports_schema = StructType([
    StructField("destination_port_code", StringType(), False),
    StructField("destination_port_name_cfg", StringType(), False),
    StructField("service_latitude", DoubleType(), False),
    StructField("service_longitude", DoubleType(), False),
])

ports_df = spark.createDataFrame(ports_data, ports_schema)
display(ports_df)


destination_port_code,destination_port_name_cfg,service_latitude,service_longitude
ESVLC,Valencia,39.430018,-0.309838
ESBCN,Barcelona,41.312166,2.209423
ESALG,Algeciras,36.166941,-5.412677


In [0]:
# ============================================================
# 2. CARGA SILVER
# ============================================================

required_tables = [positions_table, static_table]
missing_tables = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing_tables: raise RuntimeError("Faltan tablas necesarias: " + ", ".join(missing_tables))

start_date = F.to_date(F.lit(bronze_start_date))

positions = spark.table(positions_table).filter(F.to_date("bronze_partition_date") >= start_date).filter(F.col("event_timestamp").isNotNull()).filter(F.col("latitude").between(-90.0, 90.0)).filter(F.col("longitude").between(-180.0, 180.0)).dropDuplicates(["kafka_partition", "kafka_offset"])
static = spark.table(static_table).filter(F.to_date("bronze_partition_date") >= start_date).dropDuplicates(["kafka_partition", "kafka_offset"])

display(positions.agg(F.count("*").alias("positions"), F.countDistinct("mmsi").alias("vessels"), F.min("event_timestamp").alias("first_event"), F.max("event_timestamp").alias("last_event")))


positions,vessels,first_event,last_event
1689427,479,2026-08-01T20:56:49.124389Z,2026-08-24T21:08:34.821873Z


In [0]:
# ============================================================
# 3. STATIC VIGENTE + AS-OF JOIN
# ============================================================

static_time = static.withColumn("static_valid_from", F.coalesce("kafka_ingestion_timestamp", "bronze_ingested_timestamp", "silver_processed_timestamp")).filter(F.col("static_valid_from").isNotNull())
w_static_ts = Window.partitionBy("mmsi", "static_valid_from").orderBy(F.col("kafka_offset").desc_nulls_last())
static_time = static_time.withColumn("_rn", F.row_number().over(w_static_ts)).filter(F.col("_rn") == 1).drop("_rn")
w_static = Window.partitionBy("mmsi").orderBy("static_valid_from", "kafka_offset")
static_intervals = static_time.withColumn("static_valid_to", F.lead("static_valid_from").over(w_static))

p, s = positions.alias("p"), static_intervals.alias("s")
join_condition = (F.col("p.mmsi") == F.col("s.mmsi")) & (F.col("p.event_timestamp") >= F.col("s.static_valid_from")) & (F.col("s.static_valid_to").isNull() | (F.col("p.event_timestamp") < F.col("s.static_valid_to")))

enriched = p.join(s, join_condition, "left").select(
    F.col("p.mmsi").alias("mmsi"),
    F.col("p.event_timestamp").alias("event_timestamp"),
    F.col("p.latitude").alias("latitude"),
    F.col("p.longitude").alias("longitude"),
    F.col("p.speed_over_ground_knots").alias("speed_over_ground_knots"),
    F.col("p.course_over_ground_degrees").alias("course_over_ground_degrees"),
    F.col("p.true_heading_degrees").alias("true_heading_degrees"),
    F.col("p.navigation_status_code").alias("navigation_status_code"),
    F.col("p.correlation_id").alias("position_correlation_id"),
    F.col("s.imo").alias("imo"),
    F.col("s.vessel_name").alias("vessel_name"),
    F.col("s.call_sign").alias("call_sign"),
    F.col("s.ship_type_code").alias("ship_type_code"),
    F.col("s.vessel_length_meters").alias("vessel_length_meters"),
    F.col("s.vessel_beam_meters").alias("vessel_beam_meters"),
    F.col("s.vessel_draught_meters").alias("vessel_draught_meters"),
    F.col("s.destination_raw").alias("destination_raw"),
    F.col("s.destination_port_code").alias("destination_port_code"),
    F.col("s.correlation_id").alias("static_correlation_id"),
).join(F.broadcast(ports_df), "destination_port_code", "inner")


In [0]:
# ============================================================
# 4. DISTANCIA AL PUNTO DE SERVICIO + APROXIMACION
# ============================================================

lat1, lon1 = F.radians("latitude"), F.radians("longitude")
lat2, lon2 = F.radians("service_latitude"), F.radians("service_longitude")
dlat, dlon = lat2 - lat1, lon2 - lon1
a = F.pow(F.sin(dlat / 2.0), 2) + F.cos(lat1) * F.cos(lat2) * F.pow(F.sin(dlon / 2.0), 2)
distance_nm = F.lit(EARTH_RADIUS_NM) * 2.0 * F.asin(F.sqrt(F.least(F.lit(1.0), F.greatest(F.lit(0.0), a))))
bearing_deg = F.pmod(F.degrees(F.atan2(F.sin(dlon) * F.cos(lat2), F.cos(lat1) * F.sin(lat2) - F.sin(lat1) * F.cos(lat2) * F.cos(dlon))) + 360.0, 360.0)
cog_diff_deg = F.abs(F.pmod(F.col("course_over_ground_degrees") - bearing_deg + 180.0, 360.0) - 180.0)

geo = enriched.withColumn("distance_to_service_nm", distance_nm).withColumn("bearing_to_service_degrees", bearing_deg).withColumn("cog_to_service_difference_degrees", F.when(F.col("course_over_ground_degrees").between(0.0, 360.0), cog_diff_deg))

w_track = Window.partitionBy("mmsi", "destination_port_code").orderBy("event_timestamp")
track = geo.withColumn("previous_distance_nm", F.lag("distance_to_service_nm").over(w_track)).withColumn("previous_event_timestamp", F.lag("event_timestamp").over(w_track))
track = track.withColumn("distance_progress_nm", F.col("previous_distance_nm") - F.col("distance_to_service_nm")).withColumn("gap_from_previous_hours", (F.unix_timestamp("event_timestamp") - F.unix_timestamp("previous_event_timestamp")) / 3600.0)
track = track.withColumn("inside_approach_zone", F.col("distance_to_service_nm") <= APPROACH_RADIUS_NM)
track = track.withColumn("approach_signal", (F.col("inside_approach_zone") & (F.col("speed_over_ground_knots") > APPROACH_MIN_SPEED_KNOTS) & (F.col("distance_progress_nm") >= MIN_DISTANCE_PROGRESS_NM) & (F.col("cog_to_service_difference_degrees") <= MAX_COG_SERVICE_DIFF_DEG)).cast("int"))
track = track.withColumn("previous_inside_approach_zone", F.lag("inside_approach_zone").over(w_track))
track = track.withColumn("new_port_call_flag", F.when(F.col("inside_approach_zone") & (F.col("previous_inside_approach_zone").isNull() | (~F.col("previous_inside_approach_zone")) | (F.col("gap_from_previous_hours") > PORT_CALL_GAP_HOURS)), 1).otherwise(0))
track = track.withColumn("port_call_sequence", F.sum("new_port_call_flag").over(w_track.rowsBetween(Window.unboundedPreceding, Window.currentRow)))

calls_positions = track.filter(F.col("inside_approach_zone") & (F.col("port_call_sequence") > 0))
print("Posiciones en port calls:", calls_positions.count())


Posiciones en port calls: 425506


In [0]:
# ============================================================
# 5. PRIMER SERVICE
# <= 0.5 NM + SOG <= 0.2 + persistencia.
# status 5 y 15/null son compatibles; status 1 se reserva para fondeo.
# ============================================================

keys = ["mmsi", "destination_port_code", "port_call_sequence"]
w_call = Window.partitionBy(*keys).orderBy("event_timestamp")
w_call_cum = w_call.rowsBetween(Window.unboundedPreceding, Window.currentRow)

service_rows = calls_positions.withColumn("_service_signal", ((F.col("distance_to_service_nm") <= SERVICE_RADIUS_NM) & (F.col("speed_over_ground_knots") <= SERVICE_MAX_SPEED_KNOTS) & (F.col("navigation_status_code").isNull() | (F.col("navigation_status_code") != 1))).cast("int"))
service_rows = service_rows.withColumn("_prev_service_signal", F.lag("_service_signal").over(w_call)).withColumn("_next_service_signal", F.lead("_service_signal").over(w_call)).withColumn("_next_timestamp", F.lead("event_timestamp").over(w_call))
service_rows = service_rows.withColumn("_new_service_segment", F.when((F.col("_service_signal") == 1) & (F.coalesce(F.col("_prev_service_signal"), F.lit(0)) != 1), 1).otherwise(0))
service_rows = service_rows.withColumn("_service_segment_id", F.sum("_new_service_segment").over(w_call_cum))
service_rows = service_rows.withColumn("_service_interval_seconds", F.when((F.col("_service_signal") == 1) & (F.col("_next_service_signal") == 1) & ((F.unix_timestamp("_next_timestamp") - F.unix_timestamp("event_timestamp")).between(0, MAX_CONTINUOUS_GAP_MINUTES * 60)), F.unix_timestamp("_next_timestamp") - F.unix_timestamp("event_timestamp")).otherwise(0))

service_segments = service_rows.filter(F.col("_service_signal") == 1).groupBy(*keys, "_service_segment_id").agg(F.min("event_timestamp").alias("service_start_timestamp"), F.max("event_timestamp").alias("service_last_timestamp"), (F.sum("_service_interval_seconds") / 60.0).alias("service_observed_minutes"), F.count("*").alias("service_position_count"), F.avg("distance_to_service_nm").alias("service_avg_distance_nm"), F.avg("speed_over_ground_knots").alias("service_avg_speed_knots"), F.max((F.col("navigation_status_code") == 5).cast("int")).alias("service_has_status_5"))
service_segments = service_segments.filter((F.col("service_observed_minutes") >= MIN_SERVICE_MINUTES) & (F.col("service_position_count") >= MIN_SEGMENT_POSITIONS))

w_first_service = Window.partitionBy(*keys).orderBy("service_start_timestamp")
first_service = service_segments.withColumn("_rn", F.row_number().over(w_first_service)).filter(F.col("_rn") == 1).select(*keys, "service_start_timestamp", F.col("service_last_timestamp").alias("service_end_timestamp"), (F.col("service_observed_minutes") / 60.0).alias("service_observed_hours"), "service_avg_distance_nm", "service_has_status_5")


In [0]:
# ============================================================
# 6. FONDEO SOLO ANTES DEL PRIMER SERVICE
# Explicit: status=1 dentro de 5 NM.
# Fallback: SOG<=0.5 entre 0.5 y 5 NM, persistente y sin status=5.
# ============================================================

before_service = calls_positions.join(first_service.select(*keys, "service_start_timestamp"), keys, "left").filter(F.col("service_start_timestamp").isNull() | (F.col("event_timestamp") < F.col("service_start_timestamp")))

before_service = before_service.withColumn("_anchor_method", F.when((F.col("distance_to_service_nm") <= ANCHOR_RADIUS_NM) & (F.col("speed_over_ground_knots") <= ANCHOR_MAX_SPEED_KNOTS) & (F.col("navigation_status_code") == 1), "STATUS_1").when((F.col("distance_to_service_nm") > SERVICE_RADIUS_NM) & (F.col("distance_to_service_nm") <= ANCHOR_RADIUS_NM) & (F.col("speed_over_ground_knots") <= ANCHOR_MAX_SPEED_KNOTS) & (F.col("navigation_status_code").isNull() | (~F.col("navigation_status_code").isin(1, 5))), "LOW_SPEED"))
before_service = before_service.withColumn("_anchor_signal", F.col("_anchor_method").isNotNull().cast("int"))
before_service = before_service.withColumn("_prev_anchor_signal", F.lag("_anchor_signal").over(w_call)).withColumn("_next_anchor_signal", F.lead("_anchor_signal").over(w_call)).withColumn("_next_timestamp", F.lead("event_timestamp").over(w_call))
before_service = before_service.withColumn("_new_anchor_segment", F.when((F.col("_anchor_signal") == 1) & (F.coalesce(F.col("_prev_anchor_signal"), F.lit(0)) != 1), 1).otherwise(0))
before_service = before_service.withColumn("_anchor_segment_id", F.sum("_new_anchor_segment").over(w_call_cum))
before_service = before_service.withColumn("_anchor_interval_seconds", F.when((F.col("_anchor_signal") == 1) & (F.col("_next_anchor_signal") == 1) & ((F.unix_timestamp("_next_timestamp") - F.unix_timestamp("event_timestamp")).between(0, MAX_CONTINUOUS_GAP_MINUTES * 60)), F.unix_timestamp("_next_timestamp") - F.unix_timestamp("event_timestamp")).otherwise(0))

anchor_segments = before_service.filter(F.col("_anchor_signal") == 1).groupBy(*keys, "_anchor_segment_id").agg(F.min("event_timestamp").alias("anchor_start_timestamp"), F.max("event_timestamp").alias("anchor_last_timestamp"), (F.sum("_anchor_interval_seconds") / 60.0).alias("anchor_observed_minutes"), F.count("*").alias("anchor_position_count"), F.avg("distance_to_service_nm").alias("anchor_avg_distance_nm"), F.avg("speed_over_ground_knots").alias("anchor_avg_speed_knots"), F.concat_ws("+", F.sort_array(F.collect_set("_anchor_method"))).alias("anchor_detection_method"))
anchor_segments = anchor_segments.filter((F.col("anchor_observed_minutes") >= MIN_ANCHOR_MINUTES) & (F.col("anchor_position_count") >= MIN_SEGMENT_POSITIONS))

w_first_anchor = Window.partitionBy(*keys).orderBy("anchor_start_timestamp")
first_anchor = anchor_segments.withColumn("_rn", F.row_number().over(w_first_anchor)).filter(F.col("_rn") == 1).select(*keys, "anchor_start_timestamp", F.col("anchor_last_timestamp").alias("anchor_end_timestamp"), (F.col("anchor_observed_minutes") / 60.0).alias("anchor_observed_hours"), "anchor_avg_distance_nm", "anchor_detection_method")


In [0]:
# ============================================================
# 7. DIRECT_TO_SERVICE VS UNKNOWN
# ============================================================

last_before_service = calls_positions.join(first_service.select(*keys, "service_start_timestamp"), keys, "inner").filter(F.col("event_timestamp") < F.col("service_start_timestamp"))
w_last_service = Window.partitionBy(*keys).orderBy(F.col("event_timestamp").desc())
last_before_service = last_before_service.withColumn("_rn", F.row_number().over(w_last_service)).filter(F.col("_rn") == 1).select(*keys, F.col("event_timestamp").alias("last_position_timestamp"), F.col("speed_over_ground_knots").alias("last_speed_knots"), F.col("distance_to_service_nm").alias("last_distance_nm"))

call_state = first_service.join(first_anchor, keys, "full").join(last_before_service, keys, "left")
call_state = call_state.withColumn("service_entry_gap_minutes", (F.unix_timestamp("service_start_timestamp") - F.unix_timestamp("last_position_timestamp")) / 60.0)
call_state = call_state.withColumn("port_call_flow", F.when(F.col("service_start_timestamp").isNull(), "NO_SERVICE").when(F.col("anchor_start_timestamp").isNotNull(), "ANCHOR_TO_SERVICE").when(F.col("last_position_timestamp").isNotNull() & (F.col("service_entry_gap_minutes") <= DIRECT_SERVICE_MAX_GAP_MINUTES), "DIRECT_TO_SERVICE").otherwise("UNKNOWN_BEFORE_SERVICE"))
call_state = call_state.withColumn("actual_wait_hours", F.when(F.col("port_call_flow") == "ANCHOR_TO_SERVICE", (F.unix_timestamp("service_start_timestamp") - F.unix_timestamp("anchor_start_timestamp")) / 3600.0).when(F.col("port_call_flow") == "DIRECT_TO_SERVICE", F.lit(0.0)))


In [0]:
# ============================================================
# 8. SNAPSHOT AIS PREVIO A LA OPERACION
# Nombres fieles a Silver; internamente sigue siendo la ultima
# posicion inmediatamente anterior a anchor/service.
# ============================================================

cutoff = call_state.select(*keys, F.coalesce("anchor_start_timestamp", "service_start_timestamp").alias("_operation_start_timestamp"))
before_operation = calls_positions.join(cutoff, keys, "left").filter(F.col("_operation_start_timestamp").isNull() | (F.col("event_timestamp") < F.col("_operation_start_timestamp")))

approach_summary = before_operation.groupBy(*keys).agg(F.min(F.when(F.col("approach_signal") == 1, F.col("event_timestamp"))).alias("approach_start_timestamp"), F.max("approach_signal").alias("has_approach_evidence"), F.avg(F.when(F.col("approach_signal") == 1, F.col("speed_over_ground_knots"))).alias("approach_avg_speed_over_ground_knots"), F.avg(F.when(F.col("approach_signal") == 1, F.col("cog_to_service_difference_degrees"))).alias("approach_avg_cog_diff_degrees"))

w_pre = Window.partitionBy(*keys).orderBy(F.col("event_timestamp").desc())
feature_snapshot = before_operation.withColumn("_rn", F.row_number().over(w_pre)).filter(F.col("_rn") == 1).select(*keys, "event_timestamp", "speed_over_ground_knots", "course_over_ground_degrees", "true_heading_degrees", "navigation_status_code", "distance_to_service_nm")

In [0]:
# ============================================================
# 9. RESUMEN PORT CALL + BUQUE
# ============================================================

call_summary = calls_positions.groupBy(*keys).agg(
    F.min("event_timestamp").alias("port_call_start_timestamp"),
    F.max("event_timestamp").alias("port_call_end_timestamp"),
    F.count("*").alias("port_call_position_count"),
    F.min("distance_to_service_nm").alias("min_distance_to_service_nm"),
    F.max("distance_to_service_nm").alias("max_distance_to_service_nm"),
    F.first("destination_port_name_cfg", ignorenulls=True).alias("destination_port_name"),
    F.first("destination_raw", ignorenulls=True).alias("destination_raw"),
    F.first("service_latitude", ignorenulls=True).alias("service_latitude"),
    F.first("service_longitude", ignorenulls=True).alias("service_longitude"),
    F.first("imo", ignorenulls=True).alias("imo"),
    F.first("vessel_name", ignorenulls=True).alias("vessel_name"),
    F.first("call_sign", ignorenulls=True).alias("call_sign"),
    F.first("ship_type_code", ignorenulls=True).alias("ship_type_code"),
    F.first("vessel_length_meters", ignorenulls=True).alias("vessel_length_meters"),
    F.first("vessel_beam_meters", ignorenulls=True).alias("vessel_beam_meters"),
    F.first("vessel_draught_meters", ignorenulls=True).alias("vessel_draught_meters"),
    F.first("position_correlation_id", ignorenulls=True).alias("position_correlation_id"),
    F.first("static_correlation_id", ignorenulls=True).alias("static_correlation_id"),
)

calls = call_summary.join(call_state, keys, "left").join(approach_summary, keys, "left").join(feature_snapshot, keys, "left")
calls = calls.withColumn("port_call_duration_hours", (F.unix_timestamp("port_call_end_timestamp") - F.unix_timestamp("port_call_start_timestamp")) / 3600.0)


In [0]:
# ============================================================
# 10. TIPO DE BUQUE
# ============================================================

ship_code = F.col("ship_type_code")

calls = calls.withColumn(
    "ship_type_category",
    F.when(ship_code.between(20, 29), "WIG")
     .when(ship_code == 30, "Pesca")
     .when(ship_code.isin(31, 32), "Remolque")
     .when(ship_code == 33, "Dragado / operaciones submarinas")
     .when(ship_code == 34, "Operaciones de buceo")
     .when(ship_code == 35, "Operaciones militares")
     .when(ship_code == 36, "Vela")
     .when(ship_code == 37, "Recreo")
     .when(ship_code.between(40, 49), "Alta velocidad")
     .when(ship_code == 50, "Practico")
     .when(ship_code == 51, "Salvamento y rescate")
     .when(ship_code == 52, "Remolcador")
     .when(ship_code == 53, "Servicio portuario")
     .when(ship_code == 54, "Anticontaminacion")
     .when(ship_code == 55, "Fuerzas del orden")
     .when(ship_code == 58, "Transporte medico")
     .when(ship_code == 59, "No combatiente")
     .when(ship_code.between(60, 69), "Pasaje")
     .when(ship_code.between(70, 79), "Carga")
     .when(ship_code.between(80, 89), "Tanque")
     .when(ship_code.between(90, 99), "Otros")
     .when(ship_code == 0, "No disponible")
     .otherwise("Reservado"),
)


In [0]:
# ============================================================
# 11. BANDAS DE ESLORA
# ============================================================

length_m = F.col("vessel_length_meters")

calls = calls.withColumn(
    "vessel_length_band",
    F.when((length_m > 0) & (length_m < 100), "0-100")
     .when((length_m >= 100) & (length_m < 200), "100-200")
     .when((length_m >= 200) & (length_m < 300), "200-300")
     .when((length_m >= 300) & (length_m <= 600), "300-600")
     .otherwise("OTHER"),
)


In [0]:
# ============================================================
# 12. IDS, ELEGIBILIDAD INTERNA Y GOLD FINAL LIGERA
# ============================================================

imo_long = F.col("imo").cast("long")
calls = calls.withColumn("vessel_id", F.when(imo_long.isNotNull() & (imo_long > 0), F.concat(F.lit("IMO_"), imo_long.cast("string"))).otherwise(F.concat(F.lit("MMSI_"), F.col("mmsi").cast("string"))))
calls = calls.withColumn("port_call_id", F.sha2(F.concat_ws("|", "vessel_id", "destination_port_code", F.date_format("port_call_start_timestamp", "yyyy-MM-dd HH:mm:ss.SSSSSS")), 256))
calls = calls.withColumn("gold_processed_timestamp", F.current_timestamp())

# Solo se persisten columnas útiles para trazabilidad, ML y baselines.
final_df = calls.select(
    "port_call_id",
    "mmsi",
    "ship_type_category",
    "vessel_length_meters",
    "vessel_length_band",
    "vessel_beam_meters",
    "vessel_draught_meters",
    "destination_port_code",
    "event_timestamp",
    "speed_over_ground_knots",
    "course_over_ground_degrees",
    "true_heading_degrees",
    "navigation_status_code",
    "distance_to_service_nm",
    "anchor_start_timestamp",
    "service_start_timestamp",
    "actual_wait_hours",
    "gold_processed_timestamp",
)

for c in ["vessel_draught_meters", "speed_over_ground_knots", "course_over_ground_degrees", "true_heading_degrees", "distance_to_service_nm", "actual_wait_hours"]: final_df = final_df.withColumn(c, F.round(F.col(c), 3))

print("Columnas Gold persistidas:", len(final_df.columns))

Columnas Gold persistidas: 18


In [0]:
# ============================================================
# 13. ESCRITURA
# ============================================================

write_delta_table(df=final_df, target_table=target_table, target_path=target_path)

print(f"[OK] Gold creada: {target_table}")


[OK] Gold creada: masterxyz002dbr.gold.vessel_port_calls_analytics


In [0]:
# ============================================================
# 14. RESULTADO: CASOS REALES DISPONIBLES PARA ML
# La elegibilidad se calcula internamente; no se persiste como columna.
# ============================================================

result = spark.table(target_table)

# Desde la Gold ligera: actual_wait_hours no nulo identifica los casos válidos.
display(result.agg(F.count("*").alias("total_port_calls"), F.countDistinct("mmsi").alias("total_vessels"), F.sum(F.col("service_start_timestamp").isNotNull().cast("int")).alias("with_service"), F.sum(F.col("anchor_start_timestamp").isNotNull().cast("int")).alias("with_anchor"), F.sum(F.col("actual_wait_hours").isNotNull().cast("int")).alias("ml_eligible_calls"), F.countDistinct(F.when(F.col("actual_wait_hours").isNotNull(), F.col("mmsi"))).alias("ml_eligible_vessels")))

# Diagnóstico interno: sirve para validar la lógica, pero port_call_flow no está en la tabla Gold.
display(calls.groupBy("port_call_flow").agg(F.count("*").alias("port_calls"), F.countDistinct("mmsi").alias("vessels")).orderBy(F.desc("port_calls")))

display(result.filter(F.col("actual_wait_hours").isNotNull()).groupBy("destination_port_code", "vessel_length_band").agg(F.count("*").alias("port_calls"), F.countDistinct("mmsi").alias("vessels"), F.round(F.avg("actual_wait_hours"), 2).alias("avg_wait_hours"), F.round(F.expr("percentile_approx(actual_wait_hours, 0.5)"), 2).alias("median_wait_hours")).orderBy("destination_port_code", "vessel_length_band"))

total_port_calls,total_vessels,with_service,with_anchor,ml_eligible_calls,ml_eligible_vessels
609,286,259,264,202,118


port_call_flow,port_calls,vessels
null,214,164
NO_SERVICE,136,103
ANCHOR_TO_SERVICE,128,79
DIRECT_TO_SERVICE,74,67
UNKNOWN_BEFORE_SERVICE,57,49


destination_port_code,vessel_length_band,port_calls,vessels,avg_wait_hours,median_wait_hours
ESALG,100-200,10,10,10.57,1.9
ESALG,200-300,30,28,16.17,16.44
ESALG,300-600,21,18,14.05,11.83
ESBCN,100-200,22,19,10.91,0.92
ESBCN,200-300,29,26,13.56,5.04
ESBCN,300-600,13,12,7.84,0.0
ESVLC,100-200,25,24,20.04,25.67
ESVLC,200-300,31,28,16.3,16.16
ESVLC,300-600,21,20,11.63,2.12


In [0]:
# ============================================================
# 15. CONTROLES DE CALIDAD INTERNOS
# No añaden columnas a la Gold.
# ============================================================
gold = spark.table(target_table)

quality = gold.agg(
    F.count("*").alias("rows"),
    F.countDistinct("port_call_id").alias("unique_ids"),
    F.sum(
        (
            F.col("anchor_start_timestamp").isNotNull()
            & F.col("service_start_timestamp").isNotNull()
            & (F.col("service_start_timestamp") <= F.col("anchor_start_timestamp"))
        ).cast("int")
    ).alias("invalid_chronology"),
    F.sum(F.col("actual_wait_hours").isNotNull().cast("int")).alias("ml_cases"),
).first()

if quality["rows"] != quality["unique_ids"]:
    raise RuntimeError(f"port_call_id duplicados: {quality['rows'] - quality['unique_ids']}")

if quality["invalid_chronology"] > 0:
    raise RuntimeError(f"Cronologias invalidas: {quality['invalid_chronology']}")

print(f"[OK] Calidad validada. Casos ML reales: {quality['ml_cases']}")

[OK] Calidad validada. Casos ML reales: 202


In [0]:
# ============================================================
# 16. MUESTRA DE LOS CASOS UTILIZABLES PARA ML
# ============================================================

display(result.filter(F.col("actual_wait_hours").isNotNull()).select("port_call_id", "mmsi", "destination_port_code", "ship_type_category", "vessel_length_meters", "vessel_length_band", "vessel_beam_meters", "vessel_draught_meters", "event_timestamp", "speed_over_ground_knots", "course_over_ground_degrees", "true_heading_degrees", "navigation_status_code", "distance_to_service_nm", "anchor_start_timestamp", "service_start_timestamp", "actual_wait_hours").orderBy(F.col("service_start_timestamp").desc()).limit(100))

port_call_id,mmsi,destination_port_code,ship_type_category,vessel_length_meters,vessel_length_band,vessel_beam_meters,vessel_draught_meters,event_timestamp,speed_over_ground_knots,course_over_ground_degrees,true_heading_degrees,navigation_status_code,distance_to_service_nm,anchor_start_timestamp,service_start_timestamp,actual_wait_hours
f9187f9b48d171c6fab4852b69002d5c768a43f31c688b68196c8cdfc9345fc1,990976034,ESBCN,Carga,268,200-300,26,8.9,2026-08-24T18:04:44.905485Z,18.6,332.5,330,15,0.133,null,2026-08-24T18:05:42.740456Z,0.0
9c63d5018168ef8bfc4db106ae4b2a2a2e92ac352ed9ee73a0e3090875bc9495,990905083,ESBCN,Carga,295,200-300,45,9.7,2026-08-24T17:16:07.057597Z,12.0,270.5,273,15,0.072,null,2026-08-24T17:16:57.919586Z,0.0
b1bb046ca393af999ada31a63bf665beb1cef526828394832b658ad2ce66c31f,990440758,ESBCN,Carga,295,200-300,38,14.0,2026-08-24T16:48:06.731637Z,11.4,327.4,330,0,0.092,null,2026-08-24T16:48:49.442988Z,0.0
96dc0faccf4dd04e5796143960758ae4e0116e4034f931a9bd19313b7ba8919b,990938992,ESBCN,Carga,167,100-200,50,8.7,2026-08-24T15:48:08.73407Z,16.8,331.8,330,0,0.047,null,2026-08-24T15:48:57.672347Z,0.0
8b4f8be1cd12f368cd5375a51a206a95a71bbff9d22310fe0f7dee0989d19595,990889556,ESALG,Carga,211,200-300,36,15.3,2026-08-24T15:42:09.017717Z,0.0,327.2,511,1,0.0,null,2026-08-24T15:47:30.44876Z,0.0
3ca7fdc68803bd60c064522ea8412d199a5e4ac61681a1b9f6ed481c69cadcbc,990278088,ESVLC,Carga,202,200-300,34,7.1,2026-08-24T08:41:03.636542Z,18.1,229.5,511,0,0.237,2026-08-24T08:42:01.618209Z,2026-08-24T15:22:42.554603Z,6.678
b2013a4b68f5cf7ad83137f406768e053ce28a43b66600cdbf7ba986f9d886b6,990202675,ESVLC,Carga,384,300-600,41,10.1,2026-08-24T14:04:07.125655Z,0.2,271.4,511,15,0.0,2026-08-24T14:04:47.854595Z,2026-08-24T15:04:47.155605Z,1.0
82ad608a4f7f7a04602451889dbf78b67df3d9f06b68dc7b613a0dc76a51bb3a,990840069,ESALG,Carga,116,100-200,17,9.3,2026-08-24T12:04:09.698454Z,0.1,328.9,511,15,0.0,2026-08-24T12:05:01.25504Z,2026-08-24T13:59:02.73844Z,1.9
55f908b59ec4916ea49e8f5d298c806d308ceaf9e9e0e4351bd0b7e189e6cea5,990077862,ESALG,Carga,260,200-300,55,9.3,2026-08-24T10:19:39.296004Z,0.1,329.0,329,15,0.0,2026-08-24T10:20:22.024384Z,2026-08-24T12:38:34.129364Z,2.303
a0865ba515c5fac8bf9ce98f304614832c53c9bad273f9e59780c8d20d62e4d0,990995137,ESALG,Carga,330,300-600,53,10.6,2026-08-24T12:22:17.210049Z,0.1,328.8,511,1,0.0,null,2026-08-24T12:23:10.726504Z,0.0
